# Notebook Standalone — Baseline & Flip Rate

Notebook ringkas untuk menghasilkan **dua angka** yang belum ada di skripsi, **tanpa melatih ulang DeBERTa**:
1. **Baseline TF-IDF + Logistic Regression** (pembanding RQ-2) — tanpa GPU.
2. **Flip rate** (sensitivitas tanda baca, memperkuat §4.5) — memakai model yang sudah dilatih.

**Alur:** regenerasi split dataset (seed 42, identik dengan skripsi) → muat `best_model` → hitung baseline & DeBERTa pada split yang sama → hitung flip rate.

**Yang perlu kamu siapkan:**
- Attach folder `best_model` sebagai *Kaggle Dataset* (atau upload zip-nya), lalu set `MODEL_DIR` / `MODEL_ZIP` di sel Konfigurasi Model.
- Accelerator **GPU** (T4 cukup) & Internet **ON** (untuk streaming ulang dataset).

> Regenerasi 300k data via streaming makan beberapa menit (CPU); inferensi DeBERTa hanya untuk verifikasi + 2.000 teks flip rate (cepat).

## 0 — Setup lingkungan & versi

In [ ]:
# transformers 4.46.3: punya warmup_ratio (anti-NaN) & terbukti jalan di torch Colab.
# datasets 2.21 utk trust_remote_code. sentencepiece WAJIB untuk tokenizer DeBERTa-v3.
!pip -q install -U "transformers==4.46.3" "datasets==2.21.0" "accelerate>=0.34.0"     evaluate scikit-learn sentencepiece gradio
# >>> Jika versi ini SUDAH terpasang di sesi Anda, cukup lewati cell ini (tak perlu restart lagi).

In [ ]:
import os
os.environ["USE_TORCH_XLA"] = "0"   # cegah deteksi TPU/XLA yg picu error torch.xla di GPU
import re, random, unicodedata, json, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Deteksi environment otomatis (Kaggle / Colab / lokal)
def _detect_env():
    if os.path.exists("/kaggle"):
        return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except Exception:
        return "local"

ENV = _detect_env()
if ENV == "kaggle":
    OUTPUT_DIR = "/kaggle/working/skripsi_deberta_3class"
    CKPT_DIR   = "/kaggle/working/checkpoints"
elif ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/skripsi_deberta_3class"
    CKPT_DIR   = "/content/checkpoints"       # lokal (cepat), bukan Drive
else:
    OUTPUT_DIR = "./skripsi_deberta_3class"
    CKPT_DIR   = "./checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print("ENV:", ENV, "| OUTPUT_DIR:", OUTPUT_DIR, "| CKPT_DIR:", CKPT_DIR)

## 1 — Konfigurasi Model (EDIT DI SINI)

Arahkan ke folder `best_model` hasil training. Jika hanya punya zip, isi `MODEL_ZIP` — akan diekstrak otomatis.

In [ ]:
import os, zipfile

# >>> UBAH sesuai lokasi model kamu <<<
MODEL_DIR = "/kaggle/input/best-model/best_model"   # folder berisi config.json + model.safetensors + tokenizer
MODEL_ZIP = ""                                        # ex: "/kaggle/input/best-model/best_model.zip" (opsional)

if MODEL_ZIP and not (os.path.isdir(MODEL_DIR) and os.path.exists(os.path.join(MODEL_DIR, "config.json"))):
    _ext = "/kaggle/working/_model" if os.path.exists("/kaggle") else "./_model"
    os.makedirs(_ext, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP) as z:
        z.extractall(_ext)
    for _root, _dirs, _files in os.walk(_ext):
        if "config.json" in _files:
            MODEL_DIR = _root; break

assert os.path.isdir(MODEL_DIR) and os.path.exists(os.path.join(MODEL_DIR, "config.json")), \
    f"Model tidak ditemukan di {MODEL_DIR}. Attach best_model sebagai dataset atau isi MODEL_ZIP."
print("MODEL_DIR OK:", MODEL_DIR)

## 2 — Konfigurasi dataset (identik notebook utama)

In [ ]:
CONFIG = {
    "labels": ["safe", "prompt_injection", "out_of_domain"],

    # === Target ukuran (§1.3 poin 5: jumlah tidak dikunci proposal) ===
    "target_per_class": 100_000,      # ~300k total, seimbang
    "sweep_subset_size": 15_000,      # sweep di subset agar cepat; final di data penuh

    # === Sumber SAFE (instruksi/permintaan wajar, ungated) ===
    "safe_sources": [
        {"id": "Open-Orca/OpenOrca",              "text_col": "question",    "take": 60_000},
        {"id": "tatsu-lab/alpaca",                "text_col": "instruction", "take": 52_000},
        {"id": "databricks/databricks-dolly-15k", "text_col": "instruction", "take": 15_000},
        {"id": "HuggingFaceH4/no_robots",         "text_col": "prompt",      "take": 10_000},
    ],

    # === Sumber PROMPT INJECTION (ungated dulu; HackAPrompt cadangan gated) ===
    "injection_sources": [
        {"id": "jayavibhav/prompt-injection",                "text_col": "text", "label_col": "label",  "keep_label": 1, "take": 70_000},
        {"id": "imoxto/prompt_injection_cleaned_dataset-v2", "text_col": "text", "label_col": "labels", "keep_label": 1, "take": 50_000},
        {"id": "deepset/prompt-injections",                  "text_col": "text", "label_col": "label",  "keep_label": 1, "take": 400},
        # Cadangan (GATED): hanya dipakai bila target belum tercapai & Anda sudah login+akses.
        {"id": "hackaprompt/hackaprompt-dataset",            "text_col": "user_input", "take": 40_000},
    ],

    # === Sumber OUT-OF-DOMAIN (teks non-instruksi, ungated) ===
    "ood_sources": [
        {"id": "fancyzhx/ag_news",                          "text_col": "text", "take": 70_000},
        {"id": "cardiffnlp/tweet_eval", "config": "sentiment", "text_col": "text", "take": 40_000, "trust_remote_code": True},
        {"id": "cornell-movie-review-data/rotten_tomatoes", "text_col": "text", "take": 8_000},
        {"id": "google/code_x_glue_ct_code_to_text", "config": "python", "text_col": "code", "take": 15_000, "trust_remote_code": True},
    ],
    # CLINC150 OOS (sumber di proposal) — dari repo resmi clinc/oos-eval
    "clinc_json_url": "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json",

    # External test (PINT) — gated; isi manual bila tersedia
    # PINT asli GATED (minta ke Lakera). Default = file CONTOH 8 baris agar cell jalan.
    "pint_source": "https://raw.githubusercontent.com/lakeraai/pint-benchmark/main/benchmark/data/example-dataset.yaml",

    # Uji eksternal HELD-OUT (sumber BEDA dari training) untuk 3 kelas
    "heldout_per_class": 2000,
    "heldout_sources": {
        "safe":             {"id": "openai/gsm8k", "config": "main", "text_col": "question"},  # gsm8k: parquet bersih (squad gagal streaming)
        "prompt_injection": {"id": "Lakera/gandalf_ignore_instructions", "text_col": "text"},
        "out_of_domain":    {"id": "fancyzhx/dbpedia_14", "config": "dbpedia_14", "text_col": "content"},
    },

    # === Model & fine-tuning (§3.8) ===
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 192,
    "batch_size": 8,
    "grad_accum": 2,             # efektif batch 16, hemat memori
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "early_stopping_patience": 2,
    "sweep_grid": {"learning_rate": [1e-5, 2e-5, 3e-5], "epochs": [3, 4]},
    "final_epochs_cap": 4,
    # Sweep sudah dijalankan (semua konfig ~0.997-0.998). Skip agar commit muat 12 jam.
    "run_sweep": False,
    "best_params": {"learning_rate": 2e-5, "epochs": 3},  # pemenang val macro-F1 0.9982            # epoch maksimum untuk pelatihan final

    # === Opsi lain ===
    "use_class_weights": False,       # jaring pengaman; default mati
    "test_size": 0.15,
    "val_size": 0.15,
}
LABEL2ID = {l: i for i, l in enumerate(CONFIG["labels"])}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(LABEL2ID)

## 3 — Regenerasi split (seed 42) — sama persis dengan skripsi

In [ ]:
import gc
from datasets import load_dataset

def collect_class(sources, target, class_name):
    """Streaming murni tanpa shuffle (RAM nyaris 0%) + Pemotongan teks."""
    texts, srcs = [], []
    for sp in sources:
        if len(texts) >= target:
            break
        need = target - len(texts)
        take = min(sp.get("take", need), need)
        got = 0
        ds = None
        try:
            # KEMBALI KE STREAMING:
            ds = load_dataset(sp["id"], sp.get("config"), split=sp.get("split", "train"),
                              streaming=True, trust_remote_code=sp.get("trust_remote_code", False))

            # PENTING: ds.shuffle() DIHAPUS TOTAL dari sini agar tidak ada memory leak

            lab_col, keep = sp.get("label_col"), sp.get("keep_label")
            for ex in ds:
                if keep is not None:
                    try:
                        if int(ex.get(lab_col, -999)) != keep:
                            continue
                    except (ValueError, TypeError):
                        continue
                t = ex.get(sp["text_col"])
                if isinstance(t, str) and len(t.strip()) >= 3:
                    # Tetap potong di 2000 karakter agar dataframe nantinya tidak gendut
                    texts.append(t.strip()[:2000])
                    srcs.append(sp["id"])
                    got += 1

                    if got >= take or len(texts) >= target:
                        break
        except Exception as e:
            print(f"  [SKIP] {sp['id']} ({type(e).__name__}: {str(e)[:90]})")
            continue
        finally:
            # Hapus referensi stream dan paksa bersihkan RAM
            del ds
            gc.collect()

        print(f"  {sp['id']:48s} +{got}")

    df = pd.DataFrame({"text": texts, "label": class_name, "source": srcs})
    print(f"{class_name}: total {len(df)} / target {target}\n")
    return df

def load_clinc_ood():
    import urllib.request
    with urllib.request.urlopen(CONFIG["clinc_json_url"]) as r:
        data = json.load(r)
    # Potong teks json maksimal 2000 karakter
    texts = [str(it[0]).strip()[:2000] for k in ["oos_train", "oos_val", "oos_test"] for it in data.get(k, [])]
    return pd.DataFrame({"text": texts, "label": "out_of_domain", "source": "clinc_oos"})

In [ ]:
import gc

tgt = CONFIG["target_per_class"]

print(">>> SAFE")
df_safe = collect_class(CONFIG["safe_sources"], tgt, "safe")
gc.collect()  # Tambahkan ini agar sisa sampah memori langsung dibersihkan sebelum ke proses selanjutnya

print(">>> PROMPT INJECTION")
df_inj = collect_class(CONFIG["injection_sources"], tgt, "prompt_injection")
gc.collect()  # Bersihkan memori lagi

print(">>> OUT-OF-DOMAIN")
df_ood = collect_class(CONFIG["ood_sources"], tgt - 1500, "out_of_domain")
df_ood = pd.concat([df_ood, load_clinc_ood()], ignore_index=True)   # + CLINC OOS
gc.collect()  # Bersihkan memori lagi

for name, d in [("safe", df_safe), ("prompt_injection", df_inj), ("out_of_domain", df_ood)]:
    print(f"{name:16s} n={len(d):7d}")

In [ ]:
df_all = pd.concat([df_safe, df_inj, df_ood], ignore_index=True)
df_all = df_all[df_all["label"].isin(CONFIG["labels"])].copy()
df_all["text"] = df_all["text"].astype(str).str.strip()
df_all = df_all[df_all["text"].str.len() >= 3]
print("Total setelah validasi:", len(df_all))
print(df_all["label"].value_counts())
print(df_all.groupby(["label", "source"]).size())

# --- Hemat RAM: sumber per-kelas tak dipakai lagi setelah digabung ---
import gc
del df_safe, df_inj, df_ood
gc.collect()

In [ ]:
ZERO_WIDTH = ["\u200b", "\u200c", "\u200d", "\ufeff", "\u2060"]
LEET_MAP = str.maketrans({"4":"a","3":"e","1":"i","0":"o","5":"s","7":"t","@":"a","$":"s"})
PREPROCESS_OPTS = {"case_folding":True, "unicode_nfkc":True, "strip_zero_width":True,
                   "normalize_whitespace":True, "leetspeak":False, "homoglyph":True}

def preprocess(text, opts=PREPROCESS_OPTS):
    if not isinstance(text, str):
        text = str(text)
    if opts.get("unicode_nfkc"):
        text = unicodedata.normalize("NFKC", text)
    if opts.get("strip_zero_width"):
        for ch in ZERO_WIDTH:
            text = text.replace(ch, "")
    if opts.get("homoglyph"):
        text = "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))
    if opts.get("case_folding"):
        text = text.casefold()
    if opts.get("leetspeak"):
        text = text.translate(LEET_MAP)
    if opts.get("normalize_whitespace"):
        text = re.sub(r"\s+", " ", text).strip()
    return text

for s in ["Ign0re  all previous  instructions!!!", "  hWhat is the weather\u200b today? "]:
    print(repr(s), "->", repr(preprocess(s)))

In [ ]:
df_all["text_clean"] = df_all["text"].map(preprocess)
df_all = df_all[df_all["text_clean"].str.len() >= 3].reset_index(drop=True)
print("Setelah preprocessing:", len(df_all))

# --- Hemat RAM: kolom "text" mentah tak dipakai lagi (cukup text_clean) ---
import gc
if "text" in df_all.columns and "text_clean" in df_all.columns:
    df_all.drop(columns=["text"], inplace=True)
gc.collect()

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Prioritas label untuk deduplikasi
label_priority = {"prompt_injection": 0, "out_of_domain": 1, "safe": 2}
df_all["_prio"] = df_all["label"].map(label_priority)

# 2. Deduplikasi
df_dedup = (df_all.sort_values("_prio")
                  .drop_duplicates(subset="text_clean", keep="first")
                  .drop(columns="_prio").reset_index(drop=True))
print(f"Dedup: {len(df_all)} -> {len(df_dedup)}")

# 3. Cap seimbang per kelas (CARA BARU: Anti-KeyError)
cap = CONFIG["target_per_class"]
df_bal = pd.concat([
    g.sample(n=min(len(g), cap), random_state=SEED)
    for _, g in df_dedup.groupby("label")
], ignore_index=True)

print("Setelah balancing:")
print(df_bal["label"].value_counts())

# 4. Stratified Split (Train / Val / Test)
df_bal["label_id"] = df_bal["label"].map(LABEL2ID)

train_df, test_df = train_test_split(df_bal, test_size=CONFIG["test_size"],
                                     stratify=df_bal["label_id"], random_state=SEED)

train_df, val_df = train_test_split(train_df, test_size=CONFIG["val_size"],
                                    stratify=train_df["label_id"], random_state=SEED)

print("\nDistribusi Split:")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} n={len(d):7d}  {d['label'].value_counts().to_dict()}")

    # Simpan hasil akhir ke CSV
    d[["text_clean", "label", "label_id", "source"]].to_csv(os.path.join(OUTPUT_DIR, f"{name}.csv"), index=False)

## 4 — Bangun data held-out (uji eksternal)

In [ ]:
# Held-out 3 kelas dari sumber BEDA training (GSM8K / gandalf / DBpedia)
N_ho = CONFIG["heldout_per_class"]
ho_frames = []
for _cls, _sp in CONFIG["heldout_sources"].items():
    _spec = dict(_sp); _spec["take"] = N_ho
    ho_frames.append(collect_class([_spec], N_ho, _cls))
df_ho = pd.concat(ho_frames, ignore_index=True)
df_ho["text_clean"] = df_ho["text"].map(preprocess)
df_ho = (df_ho[df_ho["text_clean"].str.len() >= 3]
         .drop_duplicates("text_clean").reset_index(drop=True))
df_ho["label_id"] = df_ho["label"].map(LABEL2ID)
print("Distribusi held-out:", df_ho["label"].value_counts().to_dict())

## 5 — Baseline TF-IDF + Logistic Regression (tanpa GPU)

**Catat Macro-F1 baseline (internal & eksternal) untuk tabel pembanding BAB IV.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=2000, n_jobs=-1, C=5.0)),
])
baseline.fit(train_df["text_clean"], train_df["label"])

bl_pred_int = baseline.predict(test_df["text_clean"])
bl_f1_int = f1_score(test_df["label"], bl_pred_int, average="macro")
print("=== Baseline — Internal ===")
print(classification_report(test_df["label"], bl_pred_int, digits=4))

bl_pred_ho = baseline.predict(df_ho["text_clean"])
bl_f1_ho = f1_score(df_ho["label"], bl_pred_ho, average="macro")
print("=== Baseline — Eksternal held-out ===")
print(classification_report(df_ho["label"], bl_pred_ho, digits=4, zero_division=0))
print(f"\nBaseline Macro-F1  internal={bl_f1_int:.4f}  eksternal={bl_f1_ho:.4f}")

## 6 — Muat model DeBERTa & evaluasi pada split yang sama

Hanya inferensi (tanpa training). Sekaligus memverifikasi angka skripsi (0,9991 / 0,928).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()

@torch.no_grad()
def predict_ids(texts, batch_size=64):
    """Prediksi label-id untuk daftar teks yang SUDAH bersih (text_clean/variasi)."""
    texts = list(texts)
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i+batch_size], truncation=True, max_length=CONFIG["max_length"],
                        padding=True, return_tensors="pt").to(DEVICE)
        out.extend(model(**enc).logits.argmax(-1).cpu().tolist())
    return out

# DeBERTa internal
yb_int = predict_ids(test_df["text_clean"].tolist())
deberta_f1_int = f1_score(test_df["label_id"], yb_int, average="macro")
# DeBERTa eksternal
yb_ho = predict_ids(df_ho["text_clean"].tolist())
deberta_f1_ho = f1_score(df_ho["label_id"], yb_ho, average="macro")

print("=== TABEL PEMBANDING (Macro-F1) — untuk BAB IV ===")
print(f"{'Model':30}{'Internal':>10}{'Eksternal':>11}")
print(f"{'TF-IDF + LogReg (baseline)':30}{bl_f1_int:>10.4f}{bl_f1_ho:>11.4f}")
print(f"{'DeBERTa-v3-base (usulan)':30}{deberta_f1_int:>10.4f}{deberta_f1_ho:>11.4f}")

## 7 — Flip rate (uji sensitivitas kuantitatif)

**Catat flip rate & jumlah sampel** untuk mengganti klaim kualitatif di §4.5 dengan angka terukur.

In [ ]:
import numpy as np, pandas as pd

N_sens = 500
short = test_df[test_df["text_clean"].str.split().str.len().between(2, 12)]
samp = short.sample(min(N_sens, len(short)), random_state=42).reset_index(drop=True)

strip_end = lambda s: s.rstrip(" .?!").strip()
bases = [strip_end(t) for t in samp["text_clean"]]
bases = [b for b in bases if b]           # buang yang kosong setelah strip

variants = {
    "tanpa_tanda": bases,
    "tanda_tanya": [b + "?" for b in bases],
    "tanda_seru":  [b + "!" for b in bases],
    "titik":       [b + "." for b in bases],
}
preds = {k: [ID2LABEL[i] for i in predict_ids(v)] for k, v in variants.items()}
pv = pd.DataFrame(preds)

n_flip = int((pv.nunique(axis=1) > 1).sum())
flip_rate = n_flip / len(pv)
print(f"Sampel diuji         : {len(pv)}")
print(f"Label berubah (flip) : {n_flip}  ->  flip rate = {flip_rate:.1%}")

s_none = pd.Series(preds["tanpa_tanda"]); s_bang = pd.Series(preds["tanda_seru"]); s_ask = pd.Series(preds["tanda_tanya"])
to_inj  = int(((s_none != "prompt_injection") & (s_bang == "prompt_injection")).sum())
to_safe = int(((s_none != "safe") & (s_ask == "safe")).sum())
print(f"Berubah -> prompt_injection setelah '!': {to_inj} / {len(pv)}")
print(f"Berubah -> safe setelah '?'            : {to_safe} / {len(pv)}")

pv["base"] = bases
print("\nContoh kasus flip:")
display(pv[pv[list(preds)].nunique(axis=1) > 1].head(15))